In [1]:
import pandas as pd

In [2]:
raw = pd.read_csv('S1_prep_joint.txt', sep='\t')
raw

,count,junction_aa,day,v_call,j_call,locus
0,7000,CASSPQFTGSYEQYF,0,TRBV4-3,TRBJ2-7,beta
1,3966,CASSSPIAGQSSYEQYF,0,TRBV28,TRBJ2-7,beta
2,3221,CSASLASGTGELFF,0,TRBV20-1,TRBJ2-2,beta
3,2412,CASSYGQGNQPQHF,0,TRBV6-5,TRBJ1-5,beta
4,2292,CASSWGQGVNEQYF,0,TRBV28,TRBJ2-7,beta
...,...,...,...,...,...,...
287029,2,CSVELAFF,15,TRBV29-1,TRBJ1-1,beta
287030,2,CSVGEQFF,15,TRBV29-1,TRBJ2-1,beta
287031,2,CSATDRPI,15,TRBV29-1,TRBJ2-1,beta
287032,2,CRTNEQFF,15,TRBV20-1,TRBJ2-1,beta


In [3]:
raw0 = raw[raw.day == 0]

In [4]:
raw15 = raw[raw.day == 15]

In [5]:
giana = pd.read_csv('giana.txt', sep='\t', header=0, names=['cdr3aa', 'cluster', 'v', 'j'])
giana

,cdr3aa,cluster,v,j
0,CASILDDPNTGELFF,1,TRBV12-3,TRBJ2-2
1,CASIPDRASTDTQYF,2,TRBV12-3,TRBJ2-3
2,CASIPDRASTDTQYF,2,TRBV6-1,TRBJ2-3
3,CASKEEDQIYYEQYF,3,TRBV12-4,TRBJ2-7
4,CASKEEDQIYYEQYF,3,TRBV12-3,TRBJ2-7
...,...,...,...,...
109493,CASSFLLGLIVKDRDTGELFF,16055,TRBV12-4,TRBJ2-2
109494,CASRPRRDNYSGQSSYNSPLHF,16056,TRBV6-1,TRBJ1-6
109495,CASRPRRDNYSGQSSYNSPLHF,16056,TRBV13,TRBJ1-6
109496,CASSYPNRQFWGAVEGWNEQFF,16057,TRBV6-6,TRBJ2-1


In [6]:
ismart = pd.read_csv('ismart.txt', sep='\t', header=0, names=['cdr3aa', 'v', 'j', 'cluster'])
ismart

,cdr3aa,v,j,cluster
0,CSVGTNTEAFF,TRBV29-1,TRBJ1-1,0
1,CASSYGQGGQPQHF,TRBV6-5,TRBJ1-5,1
2,CASSYGQGNQPQHF,TRBV6-5,TRBJ1-5,1
3,CASSWGQGDQPQHF,TRBV6-5,TRBJ1-5,1
4,CASSLGQGNQPQHF,TRBV6-5,TRBJ1-5,1
...,...,...,...,...
42277,CSVLPTDTQYF,TRBV29-1,TRBJ2-3,11741
42278,CSVAFQETQYF,TRBV29-1,TRBJ2-5,11742
42279,CSVAYQETQYF,TRBV29-1,TRBJ2-5,11742
42280,CSVAGDTEAFF,TRBV29-1,TRBJ1-1,11743


In [7]:
tcremp = pd.read_csv('tcremp.tsv', sep='\t', )
for idx, col in enumerate(['cdr3aa', 'v', 'j']):
    tcremp[col] = tcremp.clone_id.apply(lambda x: x.split('_')[idx].split('*')[0]) 
tcremp = tcremp.drop(columns=['clone_id']).rename(columns={'cluster_id': 'cluster'})[['cdr3aa', 'v', 'j', 'cluster']]
tcremp = tcremp[tcremp.cluster!=-1]
tcremp

,cdr3aa,v,j,cluster
10,CASSQESGETYNEQFF,TRBV4-3,TRBJ2-1,1180
32,CASSLQGDYEQYF,TRBV7-9,TRBJ2-7,0
34,CASSSGSTDTQYF,TRBV6-4,TRBJ2-3,1
35,CASSLAGASYEQYF,TRBV12-4,TRBJ2-7,2
39,CASSLGDTYNEQFF,TRBV5-1,TRBJ2-1,3
...,...,...,...,...
263746,CSGVGYTF,TRBV20-1,TRBJ1-2,2872
263747,CSVELAFF,TRBV29-1,TRBJ1-1,4905
263748,CSVGEQFF,TRBV29-1,TRBJ2-1,61
263749,CSATDRPI,TRBV29-1,TRBJ2-1,61


In [8]:
method_to_data = {'ismart': ismart, 'giana': giana, 'tcremp': tcremp}

In [9]:
for method, df in method_to_data.items():
    print(method)
    print(df.cluster.value_counts())
    print()

ismart
cluster
25      659
388     541
307     449
73      256
1997    248
       ... 
5601      2
5602      2
5604      2
849       2
0         1
Name: count, Length: 11744, dtype: int64

giana
cluster
7042     3666
14698    2046
3051     1979
3319     1904
6855     1702
         ... 
6512        2
6513        2
6515        2
6495        2
1           1
Name: count, Length: 16057, dtype: int64

tcremp
cluster
194     334
535     295
61      294
68      234
218     232
       ... 
5586      3
5588      3
5618      3
5591      3
5593      3
Name: count, Length: 6111, dtype: int64



In [10]:
def mask_sequence(seq):
    return {seq[:i] + 'X' + seq[i+1:] for i in range(len(seq))}

def mask_all_sequences(sequences):
    masked_set = set()
    for seq in sequences:
        masked_set.update(mask_sequence(seq))
    return masked_set

In [11]:
def compute_usage(df_main, df_day0, df_day15):
    df_day0 = df_day0.copy()
    df_day15 = df_day15.copy()
    df_day0['usage'] = df_day0['count'] / df_day0['count'].sum()
    df_day15['usage'] = df_day15['count'] / df_day15['count'].sum()

    def get_usage(df):
        return df.groupby(['junction_aa', 'v_call', 'j_call'])['usage'].sum().reset_index().rename(
            columns={'junction_aa': 'cdr3aa', 'v_call': 'v', 'j_call': 'j'}
        )

    usage_day0 = get_usage(df_day0).rename(columns={'usage': 'day0_usage'})
    usage_day15 = get_usage(df_day15).rename(columns={'usage': 'day15_usage'})

    df = df_main.copy()
    df = df.merge(usage_day0, on=['cdr3aa', 'v', 'j'], how='left')
    df = df.merge(usage_day15, on=['cdr3aa', 'v', 'j'], how='left')

    df[['day0_usage', 'day15_usage']] = df[['day0_usage', 'day15_usage']].fillna(0)

    return df


In [12]:
vdjdb = pd.read_csv('vdjdb.slim.txt', sep='\t')

In [13]:
vdjdb = vdjdb[(vdjdb.gene == 'TRB') & (vdjdb['antigen.species'] == 'YFV')]

In [14]:
vdjdb.cdr3

34435        CAIQDAGASYEQYF
34544    CAISDLAGEPKTQETQYF
34777       CAISESPSGALGQFF
34856       CAISEVLTGAYGYTF
35117              CANNCGFF
                ...        
84604        CSVSGERGTDTQYF
84649        CSVTGERGTDTQYF
84690       CSVVDAAPGANVLTF
84691      CSVVDAAPGGSYEQYF
84740       CSVVLVDRGADTQYF
Name: cdr3, Length: 402, dtype: object

In [15]:
def prepare_usage_table(usage_df):
    vdjdb_yfv_masks = mask_all_sequences(vdjdb.cdr3)

    usage_df['vdjdb_yfv'] = usage_df.cdr3aa.apply(lambda x: len(mask_sequence(x).intersection(vdjdb_yfv_masks)) > 0).astype(int)

    usage_df['day0_found'] = (usage_df.day0_usage > 0).astype(int)
    usage_df['day15_found'] = (usage_df.day15_usage > 0).astype(int)

    usage_df['vdjdb_yfv_day0'] = usage_df['vdjdb_yfv'] + usage_df['day0_found'] == 2
    usage_df['vdjdb_yfv_day15'] = usage_df['vdjdb_yfv'] + usage_df['day15_found'] == 2
    return usage_df

In [16]:
def prepare_cluster_stats(usage_df):
    usage_df = usage_df.copy()
    
    delta_df = usage_df[['cluster', 
                         'day0_usage', 'day0_found', 
                         'day15_usage', 'day15_found', 
                         'vdjdb_yfv', 'vdjdb_yfv_day0', 'vdjdb_yfv_day15']].groupby(
        'cluster').sum().reset_index()
    cluster_sizes = usage_df.cluster.value_counts().reset_index().rename(columns={'count': 'cluster_size' })

    delta_df['delta'] = (delta_df.day15_usage + 0.0000001) / (delta_df.day0_usage + 0.0000001)
    delta_df = delta_df.merge(cluster_sizes)
    delta_df['yfv_15_frac_in_cluster'] = delta_df.vdjdb_yfv_day15 / delta_df.cluster_size
    return delta_df

In [17]:
usage_df = compute_usage(giana, raw0, raw15)

In [18]:
tcremp_usage_df = compute_usage(tcremp, raw0, raw15)
tcremp_usage = prepare_usage_table(tcremp_usage_df)
tcremp_clusters = prepare_cluster_stats(tcremp_usage)
tcremp_clusters

,cluster,day0_usage,day0_found,day15_usage,day15_found,vdjdb_yfv,vdjdb_yfv_day0,vdjdb_yfv_day15,delta,cluster_size,yfv_15_frac_in_cluster
0,0,0.000948,25,0.000582,21,2,1,1,0.614221,36,0.027778
1,1,0.000750,6,0.000527,3,0,0,0,0.702768,6,0.000000
2,2,0.001000,43,0.000944,35,15,13,7,0.944341,70,0.100000
3,3,0.000813,30,0.000480,30,0,0,0,0.589948,52,0.000000
4,4,0.000984,61,0.000767,63,0,0,0,0.779168,115,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
6106,6106,0.000011,1,0.000017,3,0,0,0,1.494991,3,0.000000
6107,6107,0.000013,2,0.000010,2,0,0,0,0.716093,3,0.000000
6108,6108,0.000004,1,0.000008,2,0,0,0,1.973535,3,0.000000
6109,6109,0.000004,1,0.000011,3,0,0,0,2.947509,4,0.000000


In [19]:
tcremp_clusters[tcremp_clusters.cluster_size > 10].sort_values(by='yfv_15_frac_in_cluster', ascending=False)

,cluster,day0_usage,day0_found,day15_usage,day15_found,vdjdb_yfv,vdjdb_yfv_day0,vdjdb_yfv_day15,delta,cluster_size,yfv_15_frac_in_cluster
4908,4908,0.000000,0,0.000837,11,6,0,6,8375.667392,11,0.545455
516,516,0.000097,7,0.000099,9,8,5,6,1.019128,12,0.500000
1293,1293,0.000042,6,0.000112,9,7,2,6,2.676606,13,0.461538
1396,1396,0.000019,4,0.000192,11,6,1,6,10.047928,14,0.428571
137,137,0.000141,9,0.000116,8,9,4,6,0.824077,14,0.428571
...,...,...,...,...,...,...,...,...,...,...,...
741,741,0.000040,6,0.000053,10,0,0,0,1.331902,12,0.000000
825,825,0.000086,16,0.000049,10,0,0,0,0.578010,25,0.000000
824,824,0.000059,10,0.000084,11,0,0,0,1.418006,19,0.000000
823,823,0.000053,9,0.000044,8,0,0,0,0.821393,15,0.000000


In [20]:
tcremp_clusters.to_csv('tcremp_clusters_summary.csv', index=False)

In [21]:
ismart_usage_df = compute_usage(ismart, raw0, raw15)
ismart_usage = prepare_usage_table(ismart_usage_df)
ismart_clusters = prepare_cluster_stats(ismart_usage)
ismart_clusters

,cluster,day0_usage,day0_found,day15_usage,day15_found,vdjdb_yfv,vdjdb_yfv_day0,vdjdb_yfv_day15,delta,cluster_size,yfv_15_frac_in_cluster
0,0,0.000000,0,0.000004,1,0,0,0,39.066670,1,0.0
1,1,0.006227,2,0.005179,3,0,0,0,0.831737,4,0.0
2,2,0.000004,1,0.000011,3,0,0,0,2.947509,3,0.0
3,3,0.000008,2,0.000004,1,0,0,0,0.506257,3,0.0
4,4,0.000000,0,0.000008,2,0,0,0,77.133340,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
11739,11739,0.000000,0,0.000008,2,0,0,0,77.133340,2,0.0
11740,11740,0.000008,2,0.000000,0,0,0,0,0.012959,2,0.0
11741,11741,0.000000,0,0.000008,2,0,0,0,77.133340,2,0.0
11742,11742,0.000000,0,0.000008,2,0,0,0,77.133340,2,0.0


In [22]:
ismart_clusters[ismart_clusters.cluster_size > 10].sort_values(by='yfv_15_frac_in_cluster', ascending=False)

,cluster,day0_usage,day0_found,day15_usage,day15_found,vdjdb_yfv,vdjdb_yfv_day0,vdjdb_yfv_day15,delta,cluster_size,yfv_15_frac_in_cluster
4930,4930,0.000021,4,0.000166,10,9,2,8,7.872712,13,0.615385
4762,4762,0.000004,1,0.000699,29,15,0,15,178.749895,29,0.517241
1987,1987,0.000076,12,0.000086,11,13,8,9,1.124329,19,0.473684
1784,1784,0.000080,10,0.001568,33,15,3,14,19.586957,37,0.378378
491,491,0.000154,26,0.004926,71,37,11,31,31.916160,89,0.348315
...,...,...,...,...,...,...,...,...,...,...,...
1303,1303,0.000065,10,0.000057,9,0,0,0,0.882137,14,0.000000
1330,1330,0.000046,10,0.000167,13,0,0,0,3.659194,19,0.000000
1341,1341,0.000038,9,0.000025,6,0,0,0,0.650624,15,0.000000
1380,1380,0.000053,11,0.000082,11,0,0,0,1.534020,20,0.000000


In [24]:
ismart_clusters.to_csv('ismart_clusters_summary.csv', index=False)

In [ ]:
giana_clusters[giana_clusters.cluster_size > 10].sort_values(by='yfv_15_frac_in_cluster', ascending=False)

In [152]:
giana_usage = prepare_usage_table(usage_df)

In [154]:
giana_clusters = prepare_cluster_stats(giana_usage)
giana_clusters

,cluster,day0_usage,day0_found,day15_usage,day15_found,vdjdb_yfv,vdjdb_yfv_day0,vdjdb_yfv_day15,delta,cluster_size,yfv_15_frac_in_cluster
0,1,0.000000,0,0.000015,1,0,0,0,153.266680,1,0.0
1,2,0.000006,1,0.000006,1,0,0,0,0.999557,2,0.0
2,3,0.000006,1,0.000004,1,0,0,0,0.672106,2,0.0
3,4,0.000010,1,0.000010,1,0,0,0,0.999554,2,0.0
4,5,0.000011,1,0.000019,1,0,0,0,1.660137,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
16052,16053,0.000010,1,0.000019,1,0,0,0,1.988713,2,0.0
16053,16054,0.000008,1,0.000011,2,0,0,0,1.492853,2,0.0
16054,16055,0.000032,1,0.000038,1,0,0,0,1.175398,2,0.0
16055,16056,0.000008,1,0.000004,1,0,0,0,0.506257,2,0.0


In [156]:
giana_clusters[giana_clusters.cluster_size > 10].sort_values(by='yfv_15_frac_in_cluster', ascending=False)

,cluster,day0_usage,day0_found,day15_usage,day15_found,vdjdb_yfv,vdjdb_yfv_day0,vdjdb_yfv_day15,delta,cluster_size,yfv_15_frac_in_cluster
7016,7017,0.000000,0,0.000402,12,7,0,7,4017.033681,12,0.583333
7284,7285,0.000044,6,0.000118,15,12,2,10,2.690576,20,0.500000
2049,2050,0.000030,7,0.000069,11,13,6,8,2.244899,17,0.470588
13854,13855,0.000158,25,0.000541,35,37,18,25,3.418613,54,0.462963
11015,11016,0.000048,6,0.000057,6,9,4,5,1.199041,11,0.454545
...,...,...,...,...,...,...,...,...,...,...,...
9080,9081,0.000046,9,0.000015,3,0,0,0,0.334639,12,0.000000
9113,9114,0.000070,16,0.000067,9,0,0,0,0.945596,23,0.000000
7923,7924,0.000055,11,0.000044,7,0,0,0,0.793120,16,0.000000
7925,7926,0.000185,32,0.000108,25,0,0,0,0.587587,54,0.000000


In [157]:
giana_clusters.to_csv('giana_clusters_summary.csv', index=False)